### Building a RAG System with LangChain and FAISS 
Introduction to RAG (Retrieval-Augmented Generation)
RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

### FAISS 
https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantages:
1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics


In [3]:
import os
from dotenv import load_dotenv
import warnings

warnings.filterwarnings("ignore")

# Core
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector Store
from langchain_community.vectorstores import FAISS

# Document Loaders
from langchain_community.document_loaders import TextLoader, PyPDFLoader

load_dotenv()

True

# Data Ingestion and Processing 

In [4]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [5]:
## text splitting
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

## split the documents into chunks
chunks = text_splitter.split_documents(sample_documents)
print(chunks[0])
print(chunks[1])


page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.' metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}


In [6]:

print(f"Created {len(chunks)} chunks from {len(sample_documents)} documents")
print("\nExample chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

Created 4 chunks from 4 documents

Example chunk:
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [7]:
### load the embedding models
import os
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [8]:
# Initialize OpenAI embeddings with the latest model

embeddings=OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536
)

## Example: create a embedding for a single text
sample_text="What is machine learning"
sample_embedding=embeddings.embed_query(sample_text)
sample_embedding

[-0.005901336669921875,
 -0.00595855712890625,
 0.0005350112915039062,
 -0.0335693359375,
 0.0211944580078125,
 0.0220489501953125,
 -0.0008115768432617188,
 0.009246826171875,
 -0.0225982666015625,
 0.037872314453125,
 0.01519012451171875,
 -0.03668212890625,
 -0.033966064453125,
 0.0015401840209960938,
 0.0267486572265625,
 0.015960693359375,
 0.0007944107055664062,
 -0.0308837890625,
 0.023406982421875,
 0.0048980712890625,
 8.195638656616211e-05,
 0.032470703125,
 0.042236328125,
 -0.0292816162109375,
 0.0159759521484375,
 -0.02093505859375,
 0.0266265869140625,
 0.01239776611328125,
 0.00742340087890625,
 -0.006011962890625,
 -0.0094146728515625,
 -0.0290679931640625,
 -0.0313720703125,
 0.0275726318359375,
 0.041778564453125,
 0.0003876686096191406,
 -0.00811767578125,
 -0.0106353759765625,
 -0.0555419921875,
 0.00264739990234375,
 -0.056304931640625,
 -0.016326904296875,
 0.0301971435546875,
 0.07611083984375,
 -0.025970458984375,
 -0.04888916015625,
 -0.03607177734375,
 -0.0354

In [9]:
texts=["AI","MAchine learning","Deep Learning","Neural Network"]
batch_embeddings=embeddings.embed_documents(texts)
print(batch_embeddings[0])

[-0.0081634521484375, -0.024658203125, 0.00293731689453125, 0.025115966796875, 0.006511688232421875, -0.028228759765625, -0.005035400390625, 0.0209197998046875, -0.036895751953125, 0.012786865234375, -0.003025054931640625, -0.0201263427734375, 0.0002956390380859375, -0.032745361328125, 0.0064697265625, -0.02532958984375, -0.031036376953125, -0.054351806640625, 0.032745361328125, -0.01837158203125, 0.01666259765625, 0.04827880859375, -0.0248870849609375, 0.0143585205078125, 0.0293731689453125, 0.00406646728515625, 0.00925445556640625, 0.01336669921875, 0.00254058837890625, -0.0225830078125, 0.032135009765625, -0.0279998779296875, 0.005359649658203125, -0.038177490234375, -0.0167388916015625, 0.0143280029296875, -0.03863525390625, -0.0103607177734375, -0.0105743408203125, -0.019195556640625, 0.03204345703125, 0.01459503173828125, -0.02154541015625, 0.0160675048828125, -0.0118865966796875, 0.0014524459838867188, -0.0048370361328125, -0.0335693359375, -0.0258331298828125, 0.045623779296875

In [10]:
print(batch_embeddings[1])

[-0.0182037353515625, 0.01076507568359375, 0.0174713134765625, -0.03521728515625, 0.0374755859375, 0.026123046875, 0.0158843994140625, 0.00839996337890625, -0.01093292236328125, 0.034881591796875, -0.00662994384765625, -0.06866455078125, -0.032379150390625, 0.0089569091796875, 0.036590576171875, 0.018707275390625, 0.0162811279296875, -0.0153656005859375, 0.0206298828125, 0.01300048828125, -0.0026264190673828125, 0.037811279296875, 0.034088134765625, -0.0300445556640625, 0.020660400390625, 0.00792694091796875, 0.0361328125, 0.0377197265625, 0.01053619384765625, -0.03485107421875, -0.01092529296875, -0.0241851806640625, -0.013916015625, -0.013885498046875, 0.0119781494140625, -0.0018291473388671875, -0.027587890625, 0.02783203125, -0.04510498046875, -0.00516510009765625, -0.0197296142578125, -0.04278564453125, 0.034332275390625, 0.06817626953125, -0.020599365234375, -0.034881591796875, -0.031402587890625, -0.018829345703125, 0.0005865097045898438, 0.08343505859375, -0.055572509765625, -0

In [11]:
### Compare Embedding using cosine similarity

def compare_embeddings(text1:str,text2:str):
    """Compare semantic simialrity of 2 texts usign embeddings"""

    emb1=np.array(embeddings.embed_query(text1))
    emb2=np.array(embeddings.embed_query(text2))

    ## Calculate the simialrity score

    similarity=np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity

In [13]:
# Test semantic similarity

import numpy as np
print("\nSemantic Similarity Examples:")
print(f"'AI' vs 'Artificial Intelligence': {compare_embeddings('AI', 'Artificial Intelligence'):.3f}")


Semantic Similarity Examples:
'AI' vs 'Artificial Intelligence': 0.563


In [14]:
print(f"'Machine Learning' vs 'ML': {compare_embeddings('Machine Learning', 'ML'):.3f}")

'Machine Learning' vs 'ML': 0.461


## Create Faiss Vector Store

In [15]:
vectorstore=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vector store created with {vectorstore.index.ntotal} vectors")

Vector store created with 4 vectors


In [16]:
vectorstore

In [17]:
## Save vector tore for later use
vectorstore.save_local("faiss_index")
print("Vector store saved to 'faiss_index' directory")

Vector store saved to 'faiss_index' directory


In [18]:
## load vector store
loaded_vectorstore=FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 4 vectors


In [19]:
## Similarity Search 
query="What is deep learning"

results=vectorstore.similarity_search(query,k=3)
print(results)

[Document(id='0fa12f4b-43cc-4062-8ab9-71768d88898d', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='b8821694-fb9b-416c-8a3c-c9f8b9d4d7d0', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='d07e5e7a-03ec-454a-922d-8d317f198934', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human lang

In [20]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: What is deep learning

Top 3 similar chunks:

1. Source: Deep Learning
   Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

2. Source: ML Basics
   Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

3. Source: NLP Overview
   Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
      ...


In [21]:
### Similarity Search with score
results_with_scores=vectorstore.similarity_search_with_score(query,k=3)

print("\n\nSimilarity search with scores:")
for doc, score in results_with_scores:
    print(f"\nScore: {score:.3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}...")



Similarity search with scores:

Score: 0.556
Source: Deep Learning
Content preview: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m...

Score: 1.207
Source: ML Basics
Content preview: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being...

Score: 1.273
Source: NLP Overview
Content preview: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
...


In [22]:
### Search with metadata filtering
filter_dict={"topic":"ML"}
filtered_results=vectorstore.similarity_search(
    query,
    k=3,
    filter=filter_dict
)
print(filtered_results)

[Document(id='b8821694-fb9b-416c-8a3c-c9f8b9d4d7d0', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


# Stream Rag Using LCEL

In [23]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    streaming=True,      
)

In [24]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [25]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [26]:
prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}

Answer:
"""
)

In [27]:
get_question = RunnableLambda(lambda x: x["question"])

chain = (
    {
        "context": get_question | retriever | format_docs,
        "question": get_question,
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [28]:
for chunk in chain.stream(
    {
        "question": "What is machine learning?"
    }
):
    print(chunk, end="", flush=True)

Machine learning is a subset of artificial intelligence (AI) that enables systems to learn from data. Instead of being explicitly programmed, machine learning algorithms find patterns in data. Common types of machine learning include supervised, unsupervised, and reinforcement learning.

In [35]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# Dictionary to store chat history for each session
store = {}

In [36]:
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]

In [37]:
from langchain_core.runnables.history import RunnableWithMessageHistory

with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

In [38]:
for chunk in with_history.stream(
    {
        "question": "What is machine learning?"
    },
    config={
        "configurable": {
            "session_id": "user1"
        }
    }
):
    print(chunk, end="", flush=True)

Machine learning is a subset of artificial intelligence (AI) that enables systems to learn from data. Instead of being explicitly programmed, machine learning algorithms find patterns in data. Common types of machine learning include supervised, unsupervised, and reinforcement learning.

| Feature                  | Simple RAG              | Streaming RAG                    |
| ------------------------ | ----------------------- | -------------------------------- |
| Retrieval                | ✅ Same                  | ✅ Same                           |
| Embeddings               | ✅ Same                  | ✅ Same                           |
| Vector DB (Chroma/FAISS) | ✅ Same                  | ✅ Same                           |
| Prompt                   | ✅ Same                  | ✅ Same                           |
| LLM                      | ✅ Same                  | ✅ Same                           |
| Output                   | Complete answer at once | Token-by-token as it's generated |
| LCEL Method              | `.invoke()`             | `.stream()` or `.astream()`      |
